In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
DATA_DIR = Path("C:\\Users\\attho\\OneDrive\\Desktop\\College\\ACM ATLAS\\data_centers")   # change to your actual path

In [ ]:
centers = pd.read_csv(DATA_DIR / "data_centers.csv")
timelines = pd.read_csv(DATA_DIR / "data_center_timelines.csv")
chips = pd.read_csv(DATA_DIR / "data_center_chip_quantities.csv")
chillers = pd.read_csv(DATA_DIR / "data_center_chillers.csv")
cooling = pd.read_csv(DATA_DIR / "data_center_cooling_towers.csv")
chip_types = pd.read_csv(DATA_DIR / "chip_types.csv")

In [ ]:
#inspects the csvs
datasets = {
    "centers": centers,
    "timelines": timelines,
    "chips": chips,
    "chillers": chillers,
    "cooling": cooling,
    "chip_types": chip_types
}

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

In [ ]:
# missing value check
def missing_summary(df):
    missing = df.isna().sum()
    percent = (missing / len(df) * 100).round(2)

    return (
        pd.DataFrame({
            "missing_count": missing,
            "missing_percent": percent
        })
        .query("missing_count > 0")
        .sort_values("missing_percent", ascending=False)
    )

In [ ]:
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(missing_summary(df))

In [ ]:
#relationship between tables 
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(df.head())

In [ ]:
for name, df in datasets.items():
    id_cols = [
        col for col in df.columns
        if "id" in col.lower() or "name" in col.lower()
    ]

    print(name, ":", id_cols)

In [ ]:
#join verification
center_names = set(centers["Name"].dropna())
timeline_names = set(timelines["Data center"].dropna())
chip_center_names = set(chips["Data center"].dropna())

In [ ]:
print("Centers:", len(center_names))
print("Timeline centers:", len(timeline_names))
print("Chip centers:", len(chip_center_names))

In [ ]:
print("\nTimeline centers missing from centers:")
print(timeline_names - center_names)

In [ ]:
print("\nChip centers missing from centers:")
print(chip_center_names - center_names)

In [ ]:
print("\nCenters without timeline records:")
print(center_names - timeline_names)

In [ ]:
print("\nCenters without chip records:")
print(center_names - chip_center_names)

In [ ]:
#chip type matching 
used_chips = set(chips["Chip type"].dropna())
known_chips = set(chip_types["Name"].dropna())

In [ ]:
print("Chip types used:", len(used_chips))
print("Chip types in lookup:", len(known_chips))

In [ ]:
print("\nUsed chip types missing from chip_types.csv:")
print(used_chips - known_chips)

In [ ]:
#duplicate and cardinality check
print("Unique centers:", centers["Name"].nunique())
print("Center rows:", len(centers))

In [ ]:
print("\nTimeline records per center:")
print(timelines["Data center"].value_counts().describe())

In [ ]:
print("\nChip records per center:")
print(chips["Data center"].value_counts().describe())

In [ ]:
print("\nChip types unique:", chip_types["Name"].nunique())
print("Chip type rows:", len(chip_types))

In [ ]:
print("Duplicate center names:",
      centers["Name"].duplicated().sum())

In [ ]:
print("Duplicate chip-type names:",
      chip_types["Name"].duplicated().sum())

In [ ]:
print("Exact duplicate timeline rows:",
      timelines.duplicated().sum())

In [ ]:
print("Exact duplicate chip rows:",
      chips.duplicated().sum())

In [ ]:
#specific check on water use
timelines["water_use_missing"] = timelines["Water use (MGD)"].isna()

In [ ]:
pd.crosstab(
    timelines["Construction status"],
    timelines["water_use_missing"],
    normalize="index"
)

In [ ]:
#chip table combine
chips_enriched = chips.merge(
    chip_types,
    left_on="Chip type",
    right_on="Name",
    how="left",
    validate="many_to_one",
    suffixes=("", "_chip")
)

In [ ]:
print("Before:", chips.shape)
print("After:", chips_enriched.shape)

In [ ]:
assert len(chips_enriched) == len(chips)

In [ ]:
print(
    chips_enriched[
        ["Data center", "Date", "Chip type",
         "Number of Units", "H100e", "Release date"]
    ].head()
)

In [ ]:
#temporal eda
timelines["Date"] = pd.to_datetime(
    timelines["Date"],
    errors="coerce"
)

In [ ]:
chips_enriched["Date"] = pd.to_datetime(
    chips_enriched["Date"],
    errors="coerce"
)

In [ ]:
chips_enriched["Timeline"] = pd.to_datetime(
    chips_enriched["Timeline"],
    errors="coerce"
)

In [ ]:
chips_enriched["Release date"] = pd.to_datetime(
    chips_enriched["Release date"],
    errors="coerce"
)

In [ ]:
print("Bad timeline dates:",
      timelines["Date"].isna().sum())

In [ ]:
print("Bad chip dates:",
      chips_enriched["Date"].isna().sum())

In [ ]:
print("Bad chip timeline dates:",
      chips_enriched["Timeline"].isna().sum())

In [ ]:
timelines["Year"] = timelines["Date"].dt.year
chips_enriched["Year"] = chips_enriched["Date"].dt.year

In [ ]:
print("\nTimeline year distribution:")
print(
    timelines["Year"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("\nChip year distribution:")
print(
    chips_enriched["Year"]
    .value_counts()
    .sort_index()
)

In [ ]:
#unique facilities per year
facilities_per_year = (
    timelines
    .groupby("Year")["Data center"]
    .nunique()
)

In [ ]:
print(facilities_per_year)

In [ ]:
#per facility temporal coverage 
temporal_coverage = (
    timelines
    .groupby("Data center")["Date"]
    .agg(
        first_observation="min",
        last_observation="max",
        n_observations="count"
    )
    .reset_index()
)

In [ ]:
temporal_coverage["span_years"] = (
    temporal_coverage["last_observation"]
    - temporal_coverage["first_observation"]
).dt.days / 365.25

In [ ]:
print(temporal_coverage)

In [ ]:
#construction status
print(
    timelines["Construction status"]
    .value_counts(dropna=False)
)

In [ ]:
status_by_year = pd.crosstab(
    timelines["Year"],
    timelines["Construction status"]
)

In [ ]:
print(status_by_year)

In [ ]:
#leakage check 
timeline_sorted = timelines.sort_values(
    ["Data center", "Date"]
)

In [ ]:
timeline_sorted[
    ["Data center", "Date",
     "Construction status",
     "Buildings operational",
     "Power (MW)",
     "H100 equivalents"]
].head(30)

In [ ]:
LEAKAGE_SUSPECTS = [
    "Current H100 equivalents (from Data center)"
]

In [ ]:
#temporal audit
cutoff = pd.Timestamp("2026-10-05")

In [ ]:
timelines["is_future"] = timelines["Date"] > cutoff

In [ ]:
print(timelines["is_future"].value_counts())

In [ ]:
print(
    timelines.loc[
        timelines["is_future"],
        ["Data center", "Date", "Construction status"]
    ]
    .sort_values("Date")
    .head(20)
)

In [ ]:
timelines["Year"] = timelines["Date"].dt.year
timelines["is_2023_or_earlier"] = timelines["Date"] <= "2023-12-31"

In [ ]:
#audit flag
chips_enriched["temporal_inconsistency"] = (
    chips_enriched["Date"] < chips_enriched["Release date"]
)

In [ ]:
#checking to see what datacenters are compatible with 2023
coverage_2023 = (
    timelines
    .groupby("Data center")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        has_2023_or_earlier=("is_2023_or_earlier", "max")
    )
    .reset_index()
)

In [ ]:
print(
    coverage_2023["has_2023_or_earlier"]
    .value_counts()
)

In [ ]:
#latest 2023 record
historical_2023 = (
    timelines[
        timelines["Date"] <= pd.Timestamp("2023-12-31")
    ]
    .sort_values("Date")
    .groupby("Data center")
    .tail(1)
)

In [ ]:
print("Facilities with a record by end of 2023:")
print(historical_2023["Data center"].nunique())

In [ ]:
print(
    historical_2023[
        [
            "Data center",
            "Date",
            "Construction status",
            "Buildings operational",
            "IT power (MW)",
            "Power (MW)"
        ]
    ].sort_values("Date")
)

In [ ]:
#checks when facilities first appear
first_records = (
    timelines
    .sort_values("Date")
    .groupby("Data center")
    .first()
    .reset_index()
)

In [ ]:
first_records["first_year"] = first_records["Date"].dt.year

In [ ]:
print(
    first_records["first_year"]
    .value_counts()
    .sort_index()
)

In [ ]:
print(
    first_records[
        ["Data center", "Date", "Construction status"]
    ]
    .sort_values("Date")
    .to_string(index=False)
)

In [ ]:
#chip sanity check
chips_enriched["predates_release"] = (
    chips_enriched["Date"] <
    chips_enriched["Release date"]
)

In [ ]:
print(
    chips_enriched["predates_release"]
    .value_counts(dropna=False)
)

In [ ]:
print(
    chips_enriched.loc[
        chips_enriched["predates_release"] == True,
        [
            "Data center",
            "Chip type",
            "Date",
            "Release date",
            "Number of Units"
        ]
    ].sort_values("Date")
)

In [ ]:
#Spatial relavance check
print(centers["Country"].value_counts(dropna=False))

In [ ]:
us_centers = centers[
    centers["Country"].eq("United States")
].copy()

In [ ]:
print("All Epoch centers:", len(centers))
print("US centers:", len(us_centers))

In [ ]:
#attach to timeline
timeline_geo = timelines.merge(
    centers[["Name", "Country"]],
    left_on="Data center",
    right_on="Name",
    how="left",
    validate="many_to_one"
)

In [ ]:
assert len(timeline_geo) == len(timelines)

In [ ]:
#relavant temporal coverage
us_timeline = timeline_geo.loc[
    timeline_geo["Country"] == "United States"
].copy()

In [ ]:
us_timeline["is_operational"] = (
    us_timeline["Buildings operational"] > 0
)

In [ ]:
print(
    us_timeline.groupby("Year")["Data center"]
    .nunique()
)

In [ ]:
#"operational" status tag diagnostic
us_timeline["is_operational"] = (
    us_timeline["Buildings operational"] > 0
)

In [ ]:
operational_by_year = (
    us_timeline
    .groupby("Year")
    .agg(
        facilities=("Data center", "nunique"),
        operational_records=("is_operational", "sum")
    )
)

In [ ]:
print(operational_by_year)

In [ ]:
operational_facilities = (
    us_timeline[
        us_timeline["is_operational"]
    ]
    .groupby("Year")["Data center"]
    .nunique()
)

In [ ]:
print(operational_facilities)